# Parsing and analyzing OpenQASM with the QDK

The QDK exposes its OpenQASM front end to Python, so you can inspect a program
instead of only running it. There are two layers:

- `qdk.openqasm.parser.parse` lexes and parses. It reports the structure of the
  source as written, without resolving names or types.
- `qdk.openqasm.semantic.analyze` additionally resolves identifiers, infers
  types, evaluates constants, and expands broadcast gate calls.

Both return a result rather than raising, so a program with errors still gives
you a tree to work with. This API is in preview and may change between QDK
releases.

In [ ]:
from qdk.openqasm import parser, semantic

SOURCE = """OPENQASM 3.0;
include "stdgates.inc";

const angle theta = pi / 4;

qubit[2] q;
bit[2] c;

h q[0];
ctrl @ x q[0], q[1];
rz(theta) q[1];
c = measure q;
"""

result = parser.parse(SOURCE)
print("errors:", result.has_errors)
print("version:", result.program.version)

for statement in result.program.statements:
    print(type(statement).__name__)

## Navigating a tree

Every node has named accessors for its own parts and a `children()` method for
generic traversal. There is no `kind` discriminant: dispatch with `isinstance`
or on `type(node).__name__`.

In [ ]:
gate = result.program.statements[6]

print(type(gate).__name__)
print("name:      ", gate.name)
print("modifiers: ", gate.modifiers)
print("qubits:    ", gate.qubits)
print("children:  ", [type(child).__name__ for child in gate.children()])

## Diagnostics and source positions

Spans are half-open UTF-8 byte ranges over the whole parse, including any
resolved includes. Use the result's source map to turn a span into a
line and column. A diagnostic can also render itself with the offending
source inline.

In [ ]:
broken = parser.parse("OPENQASM 3.0;\nqubit[2] q\nh q[0];\n")
source_map = broken.document.source_map

for diagnostic in broken.diagnostics:
    where = source_map.range_from_span(diagnostic.labels[0].span).start
    print(f"line {where.line}, column {where.column}: {diagnostic.message}")

print()
print(broken.diagnostics[0].render())

## Semantic analysis

`analyze` returns a different tree. Includes are resolved away, `qubit[2] q`
becomes a `QubitArrayDeclaration`, and each expression carries a resolved type
and, where the value is known at compile time, a constant value.

Resolved types are nodes too, so branch on them with `isinstance` rather than
parsing a type name.

In [ ]:
analysis = semantic.analyze(SOURCE)
print("errors:", analysis.has_errors)

for statement in analysis.program.statements:
    print(type(statement).__name__)

declaration = analysis.program.statements[0]
print()
print("declared type:", type(declaration.type).__name__)
print("is an angle:  ", isinstance(declaration.type, semantic.AngleType))
print("folded value: ", declaration.init_expr.const_value.radians)

## The symbol table

Analysis also returns the resolved symbols. The table includes everything the
program can name, so it holds the standard gates pulled in by `stdgates.inc`
alongside the program's own declarations.

In [ ]:
declared = {"theta", "q", "c"}

for symbol in analysis.symbols:
    if symbol.name in declared:
        print(f"{symbol.name}: {type(symbol.ty).__name__} ({symbol.ty.name})")

## Walking a tree with a visitor

`QASMVisitor` walks either layer. Define `visit_<ClassName>` for the nodes you
care about and call `generic_visit` to keep descending. Note that broadcast gate
calls are expanded by analysis: `h q` over a two-qubit register is one node in
the syntax tree and two in the semantic tree.

In [ ]:
from qdk.openqasm import QASMVisitor


class GateCounter(QASMVisitor):
    def __init__(self):
        self.counts = {}

    def visit_QuantumGate(self, node):
        # The semantic layer resolves the gate name to a string; the syntax
        # layer reports the `Identifier` node as written.
        name = node.name if isinstance(node.name, str) else node.name.name
        self.counts[name] = self.counts.get(name, 0) + 1
        self.generic_visit(node)


broadcast = 'OPENQASM 3.0; include "stdgates.inc"; qubit[2] q; h q;'

syntactic = GateCounter()
syntactic.visit(parser.parse(broadcast).program)
print("syntax:   ", syntactic.counts)

analyzed = GateCounter()
analyzed.visit(semantic.analyze(broadcast).program)
print("semantic: ", analyzed.counts)

A visitor can also thread a context object through the walk. Pass it to
`visit`, and every callback that declares a second parameter receives it.

In [ ]:
class QubitCollector(QASMVisitor):
    def visit_QubitArrayDeclaration(self, node, context):
        context.append((node.name, node.size.const_value))
        self.generic_visit(node, context)


registers = []
QubitCollector().visit(analysis.program, registers)
print(registers)

## Telling the two layers apart

Most class names exist in both layers, so a value named `Program` or `IntType`
does not say which tree produced it. `SyntaxNode` and `SemanticNode` answer
that at an API boundary.

In [ ]:
print("same class in both layers:  ", parser.Program is semantic.Program)
print("parsed is a SyntaxNode:     ", isinstance(result.program, parser.SyntaxNode))
print("analyzed is a SemanticNode: ", isinstance(analysis.program, semantic.SemanticNode))

## Resolving includes

`stdgates.inc`, `qelib1.inc`, and the QDK extension `qdk.inc` are built in.
Any other include is resolved through the `includes` argument, which takes a
mapping or a callback over logical `/`-separated names. Nothing falls back to
the filesystem or the network, and the resolver is not retained after the call.

In [ ]:
with_include = semantic.analyze(
    'OPENQASM 3.0;\ninclude "mylib.inc";\nqubit q;\nmygate q;\n',
    includes={"mylib.inc": "gate mygate a { U(0, 0, 0) a; }"},
)

print("errors:", with_include.has_errors)
print([type(s).__name__ for s in with_include.program.statements])

unresolved = parser.parse('OPENQASM 3.0;\ninclude "missing.inc";\n')
print("missing include:", unresolved.diagnostics[0].message)

## Canonical source and structural equality

`dumps` re-emits a syntactic program as canonical OpenQASM. It does not preserve
comments or original spelling, and it accepts only a syntax `Program`.

Nodes compare and hash structurally, so two parses of the same source are equal
and usable as `dict` keys or `set` members. Source position does not participate,
which means the same construct at two different offsets also compares equal.

In [ ]:
print(parser.dumps(result.program))

print("equal across parses:", parser.parse(SOURCE).program == result.program)

shifted = parser.parse("// a leading comment\n" + SOURCE)
print("equal after a shift:", shifted.program == result.program)

## Where to go next

- `help(qdk.openqasm.parser)` and `help(qdk.openqasm.semantic)` document every
  node class and accessor.
- The [OpenQASM interop notebook](./openqasm.ipynb) covers running, compiling,
  and estimating OpenQASM programs instead of inspecting them.